# Importation des dependences

In [61]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline



# Import des bibliothèques

In [62]:



df = pd.read_csv('../data/processed/cleaned_car_price.csv')
df.head()

,year,selling_price,km_driven,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,2007.0,60000.0,70000.0,False,False,False,True,True,False,True,False,False,False,False
1,2007.0,135000.0,50000.0,False,False,False,True,True,False,True,False,False,False,False
2,2012.0,600000.0,100000.0,True,False,False,False,True,False,True,False,False,False,False
3,2017.0,250000.0,46000.0,False,False,False,True,True,False,True,False,False,False,False
4,2014.0,450000.0,141000.0,True,False,False,False,True,False,True,False,True,False,False


#

# Séparation de variable cible

In [63]:
X = df.drop(columns=['selling_price'])
y = df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

# 1. On calcule la moyenne/écart-type ET on transforme X_train
X_train_scaled = scaler.fit_transform(X_train)

# 2. On transforme X_test avec les valeurs calculées sur X_train
X_test_scaled = scaler.transform(X_test)

print(f"taille de x_train : {X_train.shape}")
print(f"taille de x_test  : {X_test.shape}")

taille de x_train : (3388, 13)
taille de x_test  : (847, 13)


# Entrainement de modèle de base

In [64]:

# Dictionnaire regroupant les modèles à tester
models = {
    "Régression Linéaire": LinearRegression(),
    # "Ridge": Ridge(),
    # "Arbre de Décision": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    # "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(random_state=42, n_estimators=100),
    "Svr" : SVR(kernel='rbf', C=1.0, epsilon=0.1)
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({"Modèle": name, "MAE": mae, "RMSE": rmse, "R2 Score": r2})

# Conversion en DataFrame pour un affichage lisible
df_results = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
df_results

,Modèle,MAE,RMSE,R2 Score
2,XGBoost,132099.029451,192601.498311,0.594644
1,Random Forest,131770.670497,194068.625854,0.588445
0,Régression Linéaire,150076.551155,197630.240320,0.573200
3,Svr,237695.092524,313768.096750,-0.075809


# Optimisation des Hyperparamètres

In [65]:
# Identification automatique des colonnes numériques pour le StandardScaler
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ], 
    remainder='passthrough'
)
# --- Random Forest ---
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

param_grid_rf = {
    'regressor__n_estimators': [100, 200], 
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5, 10]
}

# XCBoost

pipeline_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRFRegressor(random_state=42))
])

param_grid_xgb = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [3, 6, 10],
    'regressor__learning_rate': [0.01, 0.1, 0.2],
    'regressor__subsample': [0.8, 1.0]
}

print("Optimisation de random forest en cours ")
grid_rf = GridSearchCV(
    estimator=pipeline_rf, 
    param_grid=param_grid_rf,
    cv=5, 
    scoring='r2',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Optimisation de XGBoost en cours")
grid_xgb = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid_xgb,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

# Evaluation des models optimisés et comparaison
optimized_models = {
    "Random forest (optimisé)" : grid_rf.best_estimator_,
    "XGBoot": grid_xgb.best_estimator_

}

opt_results=[]

for name, model in optimized_models.items():

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    opt_results.append({
        "Modèle": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

# Création du DataFrame des résultes optimisés

df_opt_results = pd.DataFrame(opt_results)

# Affichage des meilleurs paramètres

print("\n ==== Meilleurs HYPERPARAMETRES TROUVES  =========")
print("random forest : ", grid_rf.best_params_)
print("XGBoost : ", grid_xgb.best_params_)

# Fusion
df_final_comparison = pd.concat([df_results, df_opt_results], ignore_index=True)
df_final_comparison = df_final_comparison.sort_values(by="R2 Score", ascending=False).reset_index(drop=True)

print("\n=== TABLEAU COMPARATIF COMPLET (AVANT VS APRÈS OPTIMISATION) ===")
df_final_comparison

/home/moali/projects/carsify/venv/lib/python3.12/site-packages/xgboost/core.py:569: FutureWarning: `XGBRFRegressor` is deprecated and will be removed in a future release. The estimator is a thin wrapper over the boosting interface and does not implement a conventional random forest; features like early stopping are unsupported. Set `num_parallel_tree` along with `n_estimators=1` on the corresponding boosting estimator instead, or use a dedicated random forest implementation like those in `sklearn.ensemble`.
  return func(**kwargs)


Optimisation de random forest en cours 
Optimisation de XGBoost en cours


/home/moali/projects/carsify/venv/lib/python3.12/site-packages/xgboost/core.py:569: FutureWarning: `XGBRFRegressor` is deprecated and will be removed in a future release. The estimator is a thin wrapper over the boosting interface and does not implement a conventional random forest; features like early stopping are unsupported. Set `num_parallel_tree` along with `n_estimators=1` on the corresponding boosting estimator instead, or use a dedicated random forest implementation like those in `sklearn.ensemble`.
  return func(**kwargs)



 ==== Meilleurs HYPERPARAMETRES TROUVES  =========
random forest :  {'regressor__max_depth': 10, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 200}
XGBoost :  {'regressor__learning_rate': 0.2, 'regressor__max_depth': 10, 'regressor__n_estimators': 200, 'regressor__subsample': 1.0}

=== TABLEAU COMPARATIF COMPLET (AVANT VS APRÈS OPTIMISATION) ===


,Modèle,MAE,RMSE,R2 Score
0,Random forest (optimisé),131060.550270,186427.850447,0.620214
1,XGBoost,132099.029451,192601.498311,0.594644
2,Random Forest,131770.670497,194068.625854,0.588445
3,Régression Linéaire,150076.551155,197630.240320,0.573200
4,XGBoot,211740.395919,264526.145094,0.235364
5,Svr,237695.092524,313768.096750,-0.075809
